# EDA

`recordlinkage.datasets.load_febrl2(return_links=False)` 

[url](https://recordlinkage.readthedocs.io/en/latest/ref-datasets.html#recordlinkage.datasets.load_febrl2)

The Freely Extensible Biomedical Record Linkage (Febrl) package is distributed with a dataset generator and four datasets generated with the generator. This function returns the second Febrl dataset as a pandas.DataFrame.

> “This data set contains 5000 records (4000 originals and 1000 duplicates), with a maximum of 5 duplicates based on one original record (and a poisson distribution of duplicate records). Distribution of duplicates: 19 originals records have 5 duplicate records 47 originals records have 4 duplicate records 107 originals records have 3 duplicate records 141 originals records have 2 duplicate records 114 originals records have 1 duplicate record 572 originals records have no duplicate record”

**Goal**: Can learned pairwise similarity improve entity resolution under noisy, multi-record entities, and can graph-based refinement resolve inconsistencies in pairwise predictions?

```text
                 FEBRL-2
                    │
          5,000 noisy records
                    │
          ┌─────────┴─────────┐
          │                   │
       entities          corrupted records
          │
          ↓
      Blocking
          ↓
   Candidate pairs
          ↓
     CNN + MLP
          ↓
    pair confidence
          ↓
        Graph
          ↓
 Entity clusters
 ```


In [56]:
import recordlinkage.datasets
import pandas as pd

data = recordlinkage.datasets.load_febrl2(return_links=True)
records, links = data[0], data[1]
print(f"DataFrame: records")
print(f"Fields: {records.columns.to_list()}")
print("Description:")
print(records.info())
print("\n")

print(f"Links sie: {links.size}")
print(links[:5])
    

DataFrame: records
Fields: ['given_name', 'surname', 'street_number', 'address_1', 'address_2', 'suburb', 'postcode', 'state', 'date_of_birth', 'soc_sec_id']
Description:
<class 'pandas.DataFrame'>
Index: 5000 entries, rec-2778-org to rec-1143-org
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   given_name     4891 non-null   str   
 1   surname        4936 non-null   str   
 2   street_number  4777 non-null   object
 3   address_1      4891 non-null   str   
 4   address_2      4431 non-null   str   
 5   suburb         4950 non-null   str   
 6   postcode       5000 non-null   object
 7   state          4952 non-null   str   
 8   date_of_birth  4890 non-null   object
 9   soc_sec_id     5000 non-null   object
dtypes: object(4), str(6)
memory usage: 429.7+ KB
None


Links sie: 1934
MultiIndex([(  'rec-63-dup-1',   'rec-63-dup-0'),
            ( 'rec-232-dup-0',    'rec-232-org'),
            ('rec-3123-dup-0',

### First diagnostic

Audit FEBRL-4 specifically for the failure mode I found in previous dataset: whether true duplicates are trivially separable from non-matches by exact field agreement.

In [57]:
records = records.drop(columns='soc_sec_id')
records.sample(3)

,given_name,surname,street_number,address_1,address_2,suburb,postcode,state,date_of_birth
rec_id,,,,,,,,,
rec-3419-org,adam,blake,125,finniss crescent,fraser lodge,prospect,2620,vic,19400510
rec-2722-org,taylor,carmody,29,draper crescent,summer hill,mill park,3154,nsw,19120820
rec-3272-org,trey,held,19,holman street,kookaburra village,peregian beach,5266,nsw,19520113


In [60]:
import numpy as np
import pandas as pd

fields = records.columns.tolist()

# Count exact field agreements for every known positive
positive_exact_matches = []

for link in links:
    org_id, dup_id = link[0], link[1]
    org = records.loc[org_id]
    dup = records.loc[dup_id]

    matches = sum(
        pd.notna(org[field])
        and pd.notna(dup[field])
        and org[field] == dup[field]
        for field in fields
    )

    positive_exact_matches.append(matches)

positive_exact_matches = np.array(positive_exact_matches)

print("Positive exact-field agreement")
print("--------------------------------")
print(f"Mean   : {positive_exact_matches.mean():.2f}")
print(f"Median : {np.median(positive_exact_matches):.0f}")
print(f"Min    : {positive_exact_matches.min()}")
print(f"Max    : {positive_exact_matches.max()}")

print("\nDistribution:")
print(
    pd.Series(positive_exact_matches)
    .value_counts()
    .sort_index()
)

print("\nPercent with >= threshold:")
for threshold in range(1, 9):
    pct = (positive_exact_matches >= threshold).mean()
    print(f">= {threshold}: {pct:.2%}")

Positive exact-field agreement
--------------------------------
Mean   : 6.04
Median : 6
Min    : 1
Max    : 9

Distribution:
1      1
2      7
3     56
4    218
5    371
6    506
7    473
8    292
9     10
Name: count, dtype: int64

Percent with >= threshold:
>= 1: 100.00%
>= 2: 99.95%
>= 3: 99.59%
>= 4: 96.69%
>= 5: 85.42%
>= 6: 66.24%
>= 7: 40.07%
>= 8: 15.62%


In [61]:
# records = FEBRL-2 dataframe
# links = MultiIndex containing the known true links

rng = np.random.default_rng(42)

fields = records.columns.tolist()

# Put links into a set so we can quickly identify true matches
true_links = set(links.tolist())

record_ids = records.index.to_list()

negative_exact_matches = []

while len(negative_exact_matches) < len(links):

    id1, id2 = rng.choice(record_ids, size=2, replace=False)

    # Pairs are undirected, so check both orientations
    if (id1, id2) in true_links or (id2, id1) in true_links:
        continue

    r1 = records.loc[id1]
    r2 = records.loc[id2]

    matches = sum(
        pd.notna(r1[field])
        and pd.notna(r2[field])
        and r1[field] == r2[field]
        for field in fields
    )

    negative_exact_matches.append(matches)

negative_exact_matches = np.array(negative_exact_matches)

print("Negative exact-field agreement")
print("--------------------------------")
print(f"Mean   : {negative_exact_matches.mean():.2f}")
print(f"Median : {np.median(negative_exact_matches):.0f}")
print(f"Min    : {negative_exact_matches.min()}")
print(f"Max    : {negative_exact_matches.max()}")

print("\nDistribution:")
print(
    pd.Series(negative_exact_matches)
    .value_counts()
    .sort_index()
)

print("\nPercent with >= threshold:")
for threshold in range(1, 9):
    pct = (negative_exact_matches >= threshold).mean()
    print(f">= {threshold}: {pct:.2%}")

Negative exact-field agreement
--------------------------------
Mean   : 0.24
Median : 0
Min    : 0
Max    : 2

Distribution:
0    1477
1     449
2       8
Name: count, dtype: int64

Percent with >= threshold:
>= 1: 23.63%
>= 2: 0.41%
>= 3: 0.00%
>= 4: 0.00%
>= 5: 0.00%
>= 6: 0.00%
>= 7: 0.00%
>= 8: 0.00%


> Again... random negatives are useless for evaluating the matcher.


### hard negatives

Pairs from different entities that nevertheless have many fields in common.

```python
# Run approx 25min
import networkx as nx
from tqdm import tqdm


fields = records.columns.tolist()

# Entity graph from the FEBRL linkage
G = nx.Graph()
G.add_nodes_from(records.index)
for a, b in links:
    G.add_edge(a, b)
    
entity_id = {}
for entity_idx, component in enumerate(nx.connected_components(G)):
    for record_id in component:
        entity_id[record_id] = entity_idx
entity_id = pd.Series(entity_id)

print(f"Number of entities: {entity_id.nunique():,}")

# Convert fields to strings while preserving missingness
X = records[fields].copy()
X = X.fillna("__MISSING__")

# Find hardest cross-entity pairs
record_ids = records.index.to_numpy()
hard_negatives = []

for i in tqdm(range(len(record_ids))):
    id1 = record_ids[i]
    row1 = X.loc[id1].to_numpy()
    # Compare against records after i
    for j in range(i + 1, len(record_ids)):
        id2 = record_ids[j]
        if entity_id[id1] == entity_id[id2]:
            continue
        row2 = X.loc[id2].to_numpy()
        exact_matches = np.sum(row1 == row2)
        if exact_matches >= 3:
            hard_negatives.append((id1, id2, exact_matches))

hard_negatives = pd.DataFrame(hard_negatives, columns=["record_id_1", "record_id_2", "exact_matches"])

print("\nHard negative candidates:")
print(f"Count: {len(hard_negatives):,}")

print("\nDistribution:")
print(
    hard_negatives["exact_matches"]
    .value_counts()
    .sort_index()
)

print("\nTop hardest negatives:")
print(
    hard_negatives
    .sort_values("exact_matches", ascending=False)
    .head(30)
)
```

```console
Number of entities: 4,000
100%|██████████| 5000/5000 [21:10<00:00,  3.94it/s]  

Hard negative candidates:
Count: 12,495,566

Distribution:
exact_matches
0    9351748
1    3033620
2     108576
3       1613
4          9
Name: count, dtype: int64

Top hardest negatives:
             record_id_1     record_id_2  exact_matches
12059380  rec-3622-dup-0    rec-3570-org              4
47400       rec-3808-org    rec-3177-org              4
393765      rec-1874-org    rec-1273-org              4
1428300     rec-2249-org    rec-3445-org              4
1109399     rec-2416-org    rec-3830-org              4
6750168   rec-3447-dup-0    rec-2204-org              4
3680377     rec-3083-org    rec-1650-org              4
3374472   rec-3314-dup-1    rec-2585-org              4
3115986     rec-3830-org    rec-50-dup-1              4
10066401   rec-113-dup-0     rec-647-org              3
10932363     rec-436-org    rec-2051-org              3
783757      rec-88-dup-0    rec-2060-org              3
8841077    rec-599-dup-0    rec-2012-org              3
513976      rec-1166-org    rec-1698-org              3
12172685    rec-1954-org  rec-3677-dup-0              3
772398    rec-3752-dup-0    rec-50-dup-1              3
11132800    rec-2341-org    rec-2079-org              3
1535987      rec-488-org  rec-2916-dup-1              3
9093793     rec-1522-org    rec-3847-org              3
8814677     rec-3666-org    rec-2088-org              3
11097836    rec-3497-org    rec-3557-org              3
10831690     rec-523-org     rec-124-org              3
523672      rec-2131-org    rec-1686-org              3
10429277    rec-2171-org    rec-3126-org              3
8731027        rec-6-org     rec-465-org              3
9434863   rec-1685-dup-1    rec-2581-org              3
12105287   rec-956-dup-2    rec-3846-org              3
11680122  rec-3996-dup-1  rec-2934-dup-0              3
11624074     rec-607-org     rec-572-org              3
12347454    rec-2198-org    rec-3887-org              3
...
```

Same entity:
- 2–9 exact fields
- mean = 6.91

Different entities:
- maximum = 4 exact fields
- only 9 pairs have 4
- 1,613 pairs have 3

The distributions overlap at 3 and 4. That is already substantially better than your original dataset, where the positive and negative distributions had a huge gap. 

Most importantly: **The hardest negative is only 4/10 exact matches.** so the CNN/MLP is a meaningful approach. In comparison with the previous dataset (SPIDER v2), FEBRL-2 gives:
1. real corruption
1. variable corruption severity
1. multiple records per entity
1. duplicate-to-duplicate links
1. 4,000 latent entities
1. meaningful hard negatives
1. a natural graph-clustering problem

In [62]:
# Run approx 25min
import networkx as nx
from tqdm import tqdm


fields = records.columns.tolist()

# Entity graph from the FEBRL linkage
G = nx.Graph()
G.add_nodes_from(records.index)
for a, b in links:
    G.add_edge(a, b)
    
entity_id = {}
for entity_idx, component in enumerate(nx.connected_components(G)):
    for record_id in component:
        entity_id[record_id] = entity_idx
entity_id = pd.Series(entity_id)

print(f"Number of entities: {entity_id.nunique():,}")

# Convert fields to strings while preserving missingness
X = records[fields].copy()
X = X.fillna("__MISSING__")

# Find hardest cross-entity pairs
record_ids = records.index.to_numpy()
hard_negatives = []

for i in tqdm(range(len(record_ids))):
    id1 = record_ids[i]
    row1 = X.loc[id1].to_numpy()
    # Compare against records after i
    for j in range(i + 1, len(record_ids)):
        id2 = record_ids[j]
        if entity_id[id1] == entity_id[id2]:
            continue
        row2 = X.loc[id2].to_numpy()
        exact_matches = np.sum(row1 == row2)
        hard_negatives.append((id1, id2, exact_matches))

hard_negatives = pd.DataFrame(hard_negatives, columns=["record_id_1", "record_id_2", "exact_matches"])

print("\nHard negative candidates:")
print(f"Count: {len(hard_negatives):,}")

print("\nDistribution:")
print(
    hard_negatives["exact_matches"]
    .value_counts()
    .sort_index()
)

print("\nTop hardest negatives:")
print(
    hard_negatives
    .sort_values("exact_matches", ascending=False)
    .head(30)
)

Number of entities: 4,000


100%|██████████| 5000/5000 [21:10<00:00,  3.94it/s]  



Hard negative candidates:
Count: 12,495,566

Distribution:
exact_matches
0    9351748
1    3033620
2     108576
3       1613
4          9
Name: count, dtype: int64

Top hardest negatives:
             record_id_1     record_id_2  exact_matches
12059380  rec-3622-dup-0    rec-3570-org              4
47400       rec-3808-org    rec-3177-org              4
393765      rec-1874-org    rec-1273-org              4
1428300     rec-2249-org    rec-3445-org              4
1109399     rec-2416-org    rec-3830-org              4
6750168   rec-3447-dup-0    rec-2204-org              4
3680377     rec-3083-org    rec-1650-org              4
3374472   rec-3314-dup-1    rec-2585-org              4
3115986     rec-3830-org    rec-50-dup-1              4
10066401   rec-113-dup-0     rec-647-org              3
10932363     rec-436-org    rec-2051-org              3
783757      rec-88-dup-0    rec-2060-org              3
8841077    rec-599-dup-0    rec-2012-org              3
513976      rec-1166-org   